# Session 7: Web Development Using Flask

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + ['flask', 'requests', 'beautifulsoup4', 'lxml', 'scikit-learn', 'seaborn'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/04-application-based-programming-in-python/code/'
    for f in ['results.csv']:
        if not os.path.exists(f):
            os.makedirs(os.path.dirname(f) or '.', exist_ok=True)
            subprocess.run(['wget', '-q', '-O', f, base + f], check=False)
print('ready')

ready


## Example: `session-07_hello_flask.py`

In [2]:
"""
Session 7 · Example 1: a first Flask application - routes and URLs
------------------------------------------------------------------
Flask maps URLs to Python functions ("view functions").
This script tests the app with Flask's built-in test client, so it runs
anywhere (terminal, notebook, Colab) without opening a browser.
To serve it for real:  python session-07_hello_flask.py --serve
then open http://127.0.0.1:5000/ in a browser.
"""
import sys
from flask import Flask, request, url_for, redirect, abort

app = Flask(__name__)          # __name__ tells Flask where to find templates/static files

@app.route("/")                # the home page
def home():
    return "<h1>Welcome to the Diploma in AI portal</h1>"

@app.route("/about")
def about():
    return "This site is built with Flask, a Python micro web framework."

# Dynamic segments: <name> is passed to the function as an argument
@app.route("/hello/<name>")
def hello(name):
    return f"Assalam-o-Alaikum, {name.title()}!"

# Converters check and convert the type: int, float, path, uuid
@app.route("/square/<int:n>")
def square(n):
    return {"n": n, "square": n * n}      # returning a dict sends JSON automatically

# Query strings: /search?q=python&page=2
@app.route("/search")
def search():
    q = request.args.get("q", "")
    page = request.args.get("page", 1, type=int)
    return f"Searching for '{q}' (page {page})"

# Allowing more HTTP methods than the default GET
@app.route("/contact", methods=["GET", "POST"])
def contact():
    if request.method == "POST":
        return f"Thanks {request.form.get('name', 'friend')}, message received.", 201
    return "Contact form goes here"

@app.route("/old-courses")
def old_courses():
    return redirect(url_for("hello", name="visitor"))   # build URLs from function names

@app.route("/admin")
def admin():
    abort(403)                               # stop with an HTTP error status

@app.errorhandler(404)
def not_found(error):
    return "Sorry, that page does not exist.", 404

if __name__ == "__main__" and "--serve" in sys.argv:
    app.run(debug=True)                      # development server only
else:
    client = app.test_client()               # a fake browser for testing
    for method, url, data in [("GET", "/", None), ("GET", "/about", None),
                              ("GET", "/hello/ayesha", None), ("GET", "/square/12", None),
                              ("GET", "/square/abc", None), ("GET", "/search?q=flask&page=2", None),
                              ("POST", "/contact", {"name": "Bilal"}), ("GET", "/old-courses", None),
                              ("GET", "/admin", None), ("GET", "/nowhere", None)]:
        resp = client.open(url, method=method, data=data)
        body = resp.get_data(as_text=True).strip().replace("\n", " ")
        extra = f" -> Location: {resp.headers['Location']}" if resp.status_code in (301, 302) else ""
        print(f"{method:4} {url:<24} {resp.status_code}  {body[:60]}{extra}")
    with app.test_request_context():
        print("\nurl_for examples:", url_for("home"), url_for("hello", name="sana"),
              url_for("search", q="ai", page=3))
    print("registered routes:", sorted(r.rule for r in app.url_map.iter_rules() if r.endpoint != "static"))

GET  /                        200  <h1>Welcome to the Diploma in AI portal</h1>
GET  /about                   200  This site is built with Flask, a Python micro web framework.
GET  /hello/ayesha            200  Assalam-o-Alaikum, Ayesha!
GET  /square/12               200  {"n":12,"square":144}
GET  /square/abc              404  Sorry, that page does not exist.
GET  /search?q=flask&page=2   200  Searching for 'flask' (page 2)
POST /contact                 201  Thanks Bilal, message received.
GET  /old-courses             302  <!doctype html> <html lang=en> <title>Redirecting...</title> -> Location: /hello/visitor
GET  /admin                   403  <!doctype html> <html lang=en> <title>403 Forbidden</title> 
GET  /nowhere                 404  Sorry, that page does not exist.

url_for examples: / /hello/sana /search?q=ai&page=3
registered routes: ['/', '/about', '/admin', '/contact', '/hello/<name>', '/old-courses', '/search', '/square/<int:n>']


## Example: `session-07_templates_forms.py`

In [3]:
"""
Session 7 · Example 2: Jinja2 templates, template inheritance and HTML forms
----------------------------------------------------------------------------
Real projects keep templates as files in a templates/ folder. To keep this
example in ONE file, the templates are stored in a dictionary and loaded with
Jinja's DictLoader - the template syntax is exactly the same.
Run:  python session-07_templates_forms.py        (add --serve to open it in a browser)
"""
import sys
from flask import Flask, render_template, request, redirect, url_for, flash
from jinja2 import DictLoader

TEMPLATES = {
    "base.html": """<!doctype html>
<html><head><title>{% block title %}Portal{% endblock %}</title></head>
<body>
  <nav><a href="{{ url_for('courses') }}">Courses</a> | <a href="{{ url_for('enrol') }}">Enrol</a></nav>
  {% for msg in get_flashed_messages() %}<p class="flash">{{ msg }}</p>{% endfor %}
  {% block content %}{% endblock %}
</body></html>""",
    "courses.html": """{% extends "base.html" %}
{% block title %}Courses{% endblock %}
{% block content %}
<h1>{{ heading }}</h1>
<ul>
{% for c in courses %}
  <li>{{ loop.index }}. {{ c.title }} - {{ c.hours }} h{% if c.hours >= 36 %} <b>(long)</b>{% endif %}</li>
{% else %}
  <li>No courses yet</li>
{% endfor %}
</ul>
<p>Total: {{ courses | sum(attribute='hours') }} hours, {{ courses | length }} courses</p>
{% endblock %}""",
    "enrol.html": """{% extends "base.html" %}
{% block content %}
<h1>Enrol</h1>
{% if errors %}<ul>{% for e in errors %}
  <li class="error">{{ e }}</li>{% endfor %}
</ul>{% endif %}
<form method="post">
  <input name="name" value="{{ form.name or '' }}">
  <input name="email" value="{{ form.email or '' }}">
  <button>Submit</button>
</form>
{% endblock %}""",
}

app = Flask(__name__)
app.secret_key = "dev-only-change-me"        # needed for flash() messages (signed cookies)
app.jinja_loader = DictLoader(TEMPLATES)

COURSES = [{"title": "T-SQL", "hours": 16}, {"title": "MongoDB", "hours": 32},
           {"title": "Python Applications", "hours": 36}]
ENROLLED = []

@app.route("/")
def courses():
    return render_template("courses.html", heading="Term 1 courses", courses=COURSES)

@app.route("/enrol", methods=["GET", "POST"])
def enrol():
    errors, form = [], request.form
    if request.method == "POST":
        if len(form.get("name", "").strip()) < 3:
            errors.append("Name must have at least 3 characters")
        if "@" not in form.get("email", ""):
            errors.append("Enter a valid email")
        if not errors:
            ENROLLED.append({"name": form["name"].strip(), "email": form["email"]})
            flash(f"Welcome aboard, {form['name'].strip()}!")
            return redirect(url_for("courses"))   # Post/Redirect/Get pattern
    return render_template("enrol.html", errors=errors, form=form)

def text_only(html):
    """Strip tags and blank lines so the test output is easy to read."""
    import re
    return " | ".join(l.strip() for l in re.sub(r"<[^>]+>", "", html).splitlines() if l.strip())

if __name__ == "__main__" and "--serve" in sys.argv:
    app.run(debug=True)
else:
    c = app.test_client()
    print("GET / ->", text_only(c.get("/").get_data(as_text=True)))
    r = c.post("/enrol", data={"name": "Al", "email": "no-at-sign"})
    print("\nPOST bad form ->", r.status_code, text_only(r.get_data(as_text=True)))
    r = c.post("/enrol", data={"name": "Alishba", "email": "alishba@example.com"}, follow_redirects=True)
    print("\nPOST good form (redirect followed) ->", r.status_code, text_only(r.get_data(as_text=True))[:110])
    print("enrolled:", ENROLLED)
    # Jinja escapes HTML automatically, which blocks script injection (XSS)
    with app.test_request_context():
        from flask import render_template_string
        print("\nauto-escaping:", render_template_string("<p>{{ v }}</p>", v="<script>alert(1)</script>"))

GET / -> Courses | Courses | Enrol | Term 1 courses | 1. T-SQL - 16 h | 2. MongoDB - 32 h | 3. Python Applications - 36 h (long) | Total: 84 hours, 3 courses

POST bad form -> 200 Portal | Courses | Enrol | Enrol | Name must have at least 3 characters | Enter a valid email | Submit

POST good form (redirect followed) -> 200 Courses | Courses | Enrol | Welcome aboard, Alishba! | Term 1 courses | 1. T-SQL - 16 h | 2. MongoDB - 32 h | 
enrolled: [{'name': 'Alishba', 'email': 'alishba@example.com'}]

auto-escaping: <p>&lt;script&gt;alert(1)&lt;/script&gt;</p>


## Example: `session-07_json_api.py`

In [4]:
"""
Session 7 · Example 3: a JSON REST API with full CRUD
-----------------------------------------------------
REST maps HTTP methods to actions on a resource:
  GET /api/books        list       POST   /api/books       create
  GET /api/books/<id>   read one   PUT    /api/books/<id>  update
                                   DELETE /api/books/<id>  delete
Run:  python session-07_json_api.py      (add --serve to run a real server)
"""
import sys
from flask import Flask, jsonify, request, abort

app = Flask(__name__)
books = {1: {"id": 1, "title": "Python Crash Course", "year": 2019, "available": True},
         2: {"id": 2, "title": "Hands-On Machine Learning", "year": 2022, "available": False}}
next_id = 3

def validate(data, partial=False):
    """Return a list of problems with the submitted JSON."""
    problems = []
    if not partial or "title" in data:
        if not isinstance(data.get("title"), str) or not data["title"].strip():
            problems.append("title is required")
    if "year" in data and not (isinstance(data["year"], int) and 1450 <= data["year"] <= 2100):
        problems.append("year must be an integer between 1450 and 2100")
    return problems

@app.get("/api/books")                       # shortcut for @app.route(..., methods=["GET"])
def list_books():
    only_available = request.args.get("available") == "true"
    items = [b for b in books.values() if b["available"] or not only_available]
    return jsonify(count=len(items), books=items)

@app.get("/api/books/<int:book_id>")
def get_book(book_id):
    if book_id not in books:
        abort(404, description=f"book {book_id} not found")
    return jsonify(books[book_id])

@app.post("/api/books")
def create_book():
    global next_id
    data = request.get_json(silent=True) or {}
    problems = validate(data)
    if problems:
        return jsonify(errors=problems), 400                 # 400 Bad Request
    book = {"id": next_id, "title": data["title"].strip(), "year": data.get("year"),
            "available": data.get("available", True)}
    books[next_id] = book
    next_id += 1
    return jsonify(book), 201, {"Location": f"/api/books/{book['id']}"}   # 201 Created

@app.put("/api/books/<int:book_id>")
def update_book(book_id):
    if book_id not in books:
        abort(404, description=f"book {book_id} not found")
    data = request.get_json(silent=True) or {}
    problems = validate(data, partial=True)
    if problems:
        return jsonify(errors=problems), 400
    books[book_id].update({k: v for k, v in data.items() if k in ("title", "year", "available")})
    return jsonify(books[book_id])

@app.delete("/api/books/<int:book_id>")
def delete_book(book_id):
    if books.pop(book_id, None) is None:
        abort(404, description=f"book {book_id} not found")
    return "", 204                                           # 204 No Content

@app.errorhandler(404)
def handle_404(e):
    return jsonify(error=e.description), 404                 # JSON errors for an API

if __name__ == "__main__" and "--serve" in sys.argv:
    app.run(debug=True)
else:
    c = app.test_client()
    def call(method, url, json=None):
        r = c.open(url, method=method, json=json)
        body = r.get_json(silent=True)
        print(f"{method:6} {url:<28} {r.status_code}  {body if body is not None else '(empty)'}")
    call("GET", "/api/books")
    call("GET", "/api/books?available=true")
    call("POST", "/api/books", {"title": "Fluent Python", "year": 2022})
    call("POST", "/api/books", {"title": "  ", "year": 3000})
    call("PUT", "/api/books/2", {"available": True})
    call("DELETE", "/api/books/1")
    call("GET", "/api/books/1")
    call("GET", "/api/books")

GET    /api/books                   200  {'books': [{'available': True, 'id': 1, 'title': 'Python Crash Course', 'year': 2019}, {'available': False, 'id': 2, 'title': 'Hands-On Machine Learning', 'year': 2022}], 'count': 2}
GET    /api/books?available=true    200  {'books': [{'available': True, 'id': 1, 'title': 'Python Crash Course', 'year': 2019}], 'count': 1}
POST   /api/books                   201  {'available': True, 'id': 3, 'title': 'Fluent Python', 'year': 2022}
POST   /api/books                   400  {'errors': ['title is required', 'year must be an integer between 1450 and 2100']}
PUT    /api/books/2                 200  {'available': True, 'id': 2, 'title': 'Hands-On Machine Learning', 'year': 2022}
DELETE /api/books/1                 204  (empty)
GET    /api/books/1                 404  {'error': 'book 1 not found'}
GET    /api/books                   200  {'books': [{'available': True, 'id': 2, 'title': 'Hands-On Machine Learning', 'year': 2022}, {'available': True, 'id':

## Example: `session-07_problem_result_portal.py`

In [5]:
"""
Session 7 · Real-life problem: a student result portal
------------------------------------------------------
PROBLEM
Students keep emailing the office for their marks. Build a small Flask site that
  1. loads results.csv once at start-up,
  2. GET /                 -> a search form,
  3. GET /result?roll=...  -> an HTML result card (grades, percentage, pass/fail)
                              or a friendly 404 page for an unknown roll number,
  4. GET /api/result/<roll>-> the same data as JSON for a mobile app,
  5. GET /api/summary      -> class statistics (average per subject, top student),
  6. validates the roll-number format with a regex (Session 6) before looking it up.
Run:  python session-07_problem_result_portal.py      (add --serve for a real server)
"""
import csv
import re
import sys
from flask import Flask, jsonify, render_template_string, request, abort

ROLL_RE = re.compile(r"AI-202\d-\d{4}")
SUBJECTS = ["programming", "statistics", "databases", "communication"]

def load_results(path="results.csv"):
    """Read the CSV into {roll: record} with marks converted to int."""
    with open(path, newline="", encoding="utf-8") as f:
        return {row["roll"]: {**row, **{s: int(row[s]) for s in SUBJECTS}} for row in csv.DictReader(f)}

def grade(mark):
    for cut, g in [(85, "A"), (70, "B"), (55, "C"), (50, "D")]:
        if mark >= cut:
            return g
    return "F"

def card(rec):
    """Turn one raw record into the data shown to the student."""
    marks = {s: rec[s] for s in SUBJECTS}
    pct = sum(marks.values()) / len(marks)
    return {"roll": rec["roll"], "name": rec["name"],
            "subjects": {s: {"mark": m, "grade": grade(m)} for s, m in marks.items()},
            "percentage": round(pct, 1), "passed": all(m >= 50 for m in marks.values())}

PAGE = """<!doctype html><title>Results</title>
<h1>Result portal</h1>
<form action="/result"><input name="roll" placeholder="AI-2026-0001"><button>Check</button></form>
{% if error %}<p class="error">{{ error }}</p>{% endif %}
{% if c %}
<h2>{{ c.name }} ({{ c.roll }})</h2>
<table>{% for s, v in c.subjects.items() %}
<tr><td>{{ s|title }}</td><td>{{ v.mark }}</td><td>{{ v.grade }}</td></tr>{% endfor %}
</table>
<p>Percentage: {{ c.percentage }}% - {{ "PASSED" if c.passed else "NOT PASSED: re-sit required" }}</p>
{% endif %}"""

app = Flask(__name__)
RESULTS = load_results()

@app.get("/")
def index():
    return render_template_string(PAGE)

@app.get("/result")
def result():
    roll = request.args.get("roll", "").strip().upper()
    if not ROLL_RE.fullmatch(roll):
        return render_template_string(PAGE, error=f"'{roll}' is not a valid roll number (format AI-2026-0001)"), 400
    if roll not in RESULTS:
        return render_template_string(PAGE, error=f"No result found for {roll}"), 404
    return render_template_string(PAGE, c=card(RESULTS[roll]))

@app.get("/api/result/<roll>")
def api_result(roll):
    roll = roll.upper()
    if roll not in RESULTS:
        abort(404)
    return jsonify(card(RESULTS[roll]))

@app.get("/api/summary")
def api_summary():
    cards = [card(r) for r in RESULTS.values()]
    averages = {s: round(sum(r[s] for r in RESULTS.values()) / len(RESULTS), 1) for s in SUBJECTS}
    top = max(cards, key=lambda c: c["percentage"])
    return jsonify(students=len(cards), passed=sum(c["passed"] for c in cards),
                   subject_average=averages, top_student={"name": top["name"], "percentage": top["percentage"]})

@app.errorhandler(404)
def nf(e):
    return jsonify(error="not found"), 404

def strip(html):
    """Visible text of a page, minus the repeated header and search form."""
    import html as h
    text = " | ".join(" ".join(l.split()) for l in re.sub(r"<[^>]+>", " ", html).splitlines() if l.strip())
    return h.unescape(text.split("Check", 1)[-1].lstrip(" |"))

if __name__ == "__main__" and "--serve" in sys.argv:
    app.run(debug=True)
else:
    c = app.test_client()
    for url in ["/result?roll=ai-2026-0001", "/result?roll=AI-2026-0003", "/result?roll=AI-2026-0099",
                "/result?roll=12345"]:
        r = c.get(url)
        print(f"GET {url:<28} {r.status_code}\n   {strip(r.get_data(as_text=True))}\n")
    print("GET /api/result/AI-2026-0004 ->", c.get("/api/result/AI-2026-0004").get_json())
    print("GET /api/result/XX           ->", c.get("/api/result/XX").status_code)
    print("GET /api/summary ->", c.get("/api/summary").get_json())

GET /result?roll=ai-2026-0001    200
   Ayesha Siddiqui (AI-2026-0001) | Programming 91 A | Statistics 84 B | Databases 88 A | Communication 79 B | Percentage: 85.5% - PASSED

GET /result?roll=AI-2026-0003    200
   Mehwish Ali (AI-2026-0003) | Programming 45 F | Statistics 51 D | Databases 62 C | Communication 70 B | Percentage: 57.0% - NOT PASSED: re-sit required

GET /result?roll=AI-2026-0099    404
   No result found for AI-2026-0099

GET /result?roll=12345           400
   '12345' is not a valid roll number (format AI-2026-0001)

GET /api/result/AI-2026-0004 -> {'name': 'Danish Raza', 'passed': True, 'percentage': 89.0, 'roll': 'AI-2026-0004', 'subjects': {'communication': {'grade': 'A', 'mark': 85}, 'databases': {'grade': 'A', 'mark': 90}, 'programming': {'grade': 'A', 'mark': 88}, 'statistics': {'grade': 'A', 'mark': 93}}}
GET /api/result/XX           -> 404
GET /api/summary -> {'passed': 4, 'students': 5, 'subject_average': {'communication': 81.4, 'databases': 75.8, 'programmin

## Example: `session-07_project_notice_board.py`

In [6]:
"""
Session 7 · Mini project: campus notice board (HTML pages + JSON API + tests)
-----------------------------------------------------------------------------
Features
  - an application factory create_app() (the pattern used by real Flask projects)
  - HTML home page listing notices, newest first, filterable by category
  - JSON API: GET /api/notices, POST /api/notices (needs an API token),
    DELETE /api/notices/<id> (needs the token)
  - notices saved to a JSON file so they survive restarts (Session 5)
  - a set of automated checks using the test client, like a mini test suite
Run:  python session-07_project_notice_board.py     (add --serve to browse it)
"""
import json
import os
import sys
import tempfile
from datetime import date
from flask import Flask, abort, jsonify, render_template_string, request

PAGE = """<!doctype html><title>Notice board</title>
<h1>Campus notice board</h1>
<p>Filter: <a href="/">all</a>{% for c in categories %} | <a href="/?category={{ c }}">{{ c }}</a>{% endfor %}</p>
{% for n in notices %}
<article><h3>[{{ n.category }}] {{ n.title }}</h3><small>{{ n.date }}</small><p>{{ n.body }}</p></article>
{% else %}<p>No notices.</p>{% endfor %}"""
CATEGORIES = ["exam", "event", "holiday", "general"]

def create_app(data_file, api_token):
    """Build and configure a Flask app. Tests can create as many as they like."""
    app = Flask(__name__)
    app.config.update(DATA_FILE=data_file, API_TOKEN=api_token)

    def load():
        try:
            with open(app.config["DATA_FILE"], encoding="utf-8") as f:
                return json.load(f)
        except FileNotFoundError:
            return []

    def save(notices):
        with open(app.config["DATA_FILE"], "w", encoding="utf-8") as f:
            json.dump(notices, f, indent=2)

    def require_token():
        sent = request.headers.get("Authorization", "")
        if sent != f"Bearer {app.config['API_TOKEN']}":
            abort(401)

    def visible(category=None):
        items = sorted(load(), key=lambda n: (n["date"], n["id"]), reverse=True)
        return [n for n in items if category in (None, n["category"])]

    @app.get("/")
    def home():
        cat = request.args.get("category")
        return render_template_string(PAGE, notices=visible(cat), categories=CATEGORIES)

    @app.get("/api/notices")
    def list_notices():
        return jsonify(visible(request.args.get("category")))

    @app.post("/api/notices")
    def add_notice():
        require_token()
        data = request.get_json(silent=True) or {}
        errors = [f"{f} is required" for f in ("title", "body") if not str(data.get(f, "")).strip()]
        if data.get("category", "general") not in CATEGORIES:
            errors.append(f"category must be one of {CATEGORIES}")
        if errors:
            return jsonify(errors=errors), 400
        notices = load()
        notice = {"id": max((n["id"] for n in notices), default=0) + 1,
                  "title": data["title"].strip(), "body": data["body"].strip(),
                  "category": data.get("category", "general"),
                  "date": data.get("date", date.today().isoformat())}
        notices.append(notice)
        save(notices)
        return jsonify(notice), 201

    @app.delete("/api/notices/<int:nid>")
    def delete_notice(nid):
        require_token()
        notices = load()
        remaining = [n for n in notices if n["id"] != nid]
        if len(remaining) == len(notices):
            abort(404)
        save(remaining)
        return "", 204

    @app.errorhandler(401)
    def unauthorised(e):
        return jsonify(error="missing or wrong API token"), 401

    @app.errorhandler(404)
    def missing(e):
        return jsonify(error="not found"), 404

    return app

# ---------------------------------------------------------------------------
if __name__ == "__main__" and "--serve" in sys.argv:
    create_app("notices.json", os.environ.get("NOTICE_TOKEN", "change-me")).run(debug=True)
else:
    tmp = tempfile.NamedTemporaryFile(suffix=".json", delete=False).name
    os.remove(tmp)                                   # start with no file at all
    app = create_app(tmp, api_token="s3cret")
    client = app.test_client()
    auth = {"Authorization": "Bearer s3cret"}
    passed = failed = 0

    def check(label, condition):
        """A tiny assertion helper that reports instead of stopping."""
        global passed, failed
        passed, failed = passed + bool(condition), failed + (not condition)
        print(f"  [{'PASS' if condition else 'FAIL'}] {label}")

    print("Automated checks")
    check("empty board shows 'No notices'", "No notices" in client.get("/").get_data(as_text=True))
    r = client.post("/api/notices", json={"title": "Mid-term exams", "body": "Start 10 Nov"})
    check("POST without token is rejected with 401", r.status_code == 401)
    seed = [
        {"title": "Mid-term exams", "body": "Start 10 Nov in Hall A.", "category": "exam", "date": "2026-10-20"},
        {"title": "AI hackathon", "body": "Teams of 3, register by Friday.", "category": "event", "date": "2026-10-22"},
        {"title": "Iqbal Day", "body": "Campus closed on 9 November.", "category": "holiday", "date": "2026-10-25"},
    ]
    for n in seed:
        r = client.post("/api/notices", json=n, headers=auth)
    check("POST with token returns 201 and an id", r.status_code == 201 and r.get_json()["id"] == 3)
    r = client.post("/api/notices", json={"title": "", "category": "party"}, headers=auth)
    check("invalid notice returns 400 with 3 errors", r.status_code == 400 and len(r.get_json()["errors"]) == 3)
    items = client.get("/api/notices").get_json()
    check("list is newest first", [n["title"] for n in items] == ["Iqbal Day", "AI hackathon", "Mid-term exams"])
    check("category filter works", [n["title"] for n in client.get("/api/notices?category=exam").get_json()] == ["Mid-term exams"])
    check("HTML page shows the hackathon", "AI hackathon" in client.get("/").get_data(as_text=True))
    check("data survives a 'restart' (new app, same file)",
          len(create_app(tmp, "s3cret").test_client().get("/api/notices").get_json()) == 3)
    check("DELETE removes a notice (204)", client.delete("/api/notices/2", headers=auth).status_code == 204)
    check("DELETE of a missing id gives 404", client.delete("/api/notices/99", headers=auth).status_code == 404)
    print(f"\n{passed} passed, {failed} failed")
    print("final board:", [(n["id"], n["category"], n["title"]) for n in client.get("/api/notices").get_json()])
    os.remove(tmp)

Automated checks
  [PASS] empty board shows 'No notices'
  [PASS] POST without token is rejected with 401
  [PASS] POST with token returns 201 and an id
  [PASS] invalid notice returns 400 with 3 errors
  [PASS] list is newest first
  [PASS] category filter works
  [PASS] HTML page shows the hackathon
  [PASS] data survives a 'restart' (new app, same file)
  [PASS] DELETE removes a notice (204)
  [PASS] DELETE of a missing id gives 404

10 passed, 0 failed
final board: [(3, 'holiday', 'Iqbal Day'), (1, 'exam', 'Mid-term exams')]
